# 22. Alert Configuration & Incident Response
**Duration:** 20 min | **Topics:** SLOs, Azure Monitor alerts, runbooks

---

## 🎯 Learning Objectives

By the end of this notebook you will be able to:

| # | Skill | Why It Matters |
|---|-------|----------------|
| 1 | Define **Service Level Objectives (SLOs)** for latency, availability, error rate, and token cost | SLOs are the contract between your team and your users — without them, you have no definition of "broken" |
| 2 | Calculate **SLO burn rate** to predict when your error budget will be exhausted | Burn rate tells you whether you need to act now (10x burn) or can investigate later (1x burn) |
| 3 | Create **Azure Monitor alert rules** with `az monitor scheduled-query create` | Alerts must fire *before* users complain — setting them at 80% of your SLO target gives you that window |
| 4 | Configure **budget alerts** to cap unexpected Azure spend | A single runaway prompt loop can burn through thousands of dollars — a $40/day budget alert stops that |
| 5 | Set up **smart anomaly detection** for unusual traffic patterns | Rule-based alerts need thresholds; anomaly detection catches spikes you never thought to alert on |
| 6 | Execute an **incident response runbook** with concrete rollback commands | Having pre-written commands reduces mean time to recover (MTTR) from 30+ minutes to under 5 |

---

## 📐 Alert Architecture

```
App Insights (customEvents)
  │
  ├── Scheduled Query Alert  → error rate > 2%    → Action Group → PagerDuty email
  ├── Scheduled Query Alert  → P99 latency > 10s  → Action Group → Slack webhook
  ├── Smart Detector         → anomalous failures  → Action Group → email
  └── Budget Alert           → spend > $40/day     → Action Group → email
```

### Minimum Azure Resources
| Resource | SKU | Cost |
|---|---|---|
| Azure Monitor Alerts | - | First 1,000/month FREE |
| Action Groups (email) | - | First 1,000 emails FREE |
| Application Insights | Pay-as-you-go | 5 GB free/month |


In [ ]:
import subprocess, sys

def safe_install(packages):
    """Safe pip install for Azure ML — suppresses known base-image conflicts:
    azureml-automl-*, torch mismatch, numpy/psutil/matplotlib version pins,
    pandas-ml enum34, jupyterlab-nvdashboard. None affect notebook code."""
    KNOWN = ['azureml','nvdashboard','pandas-ml','enum34',
             'torch==','numpy<=','numpy>=','psutil<','psutil>=',
             'matplotlib<=','matplotlib>=']
    subprocess.run([sys.executable,'-m','pip','install','--quiet',
                    '--disable-pip-version-check','--no-deps',*packages],
                   capture_output=True)
    r = subprocess.run([sys.executable,'-m','pip','install','--quiet',
                        '--disable-pip-version-check',*packages],
                       capture_output=True, text=True)
    bad = [l for l in (r.stderr or '').splitlines()
           if 'ERROR' in l and not any(k in l for k in KNOWN)]
    print(f'✅ Ready: {", ".join(packages)}') if not bad else print('⚠️', bad)

safe_install(['azure-mgmt-monitor', 'azure-mgmt-consumption', 'azure-identity'])


✅ Ready: azure-mgmt-monitor, azure-mgmt-consumption, azure-identity
   (Azure ML env conflicts suppressed — safe to ignore)


## Step 1: Define SLOs for Your LLM Service

In [ ]:
import json

# SLO = what you promise users. SLA = contractual version of SLO.
# Always set alert threshold BELOW SLO target so you get warning before breach.

slos = [
    {"name": "API Availability",    "target": "99.5%",    "window": "30d",
     "alert_at": "< 99.0%",         "severity": 1,
     "why": "Below 99% = ~7hr downtime/month — unacceptable for prod"},
    {"name": "P50 Latency",         "target": "< 2000ms", "window": "1h",
     "alert_at": "> 3000ms",        "severity": 2,
     "why": "P50 >3s = majority of users having bad experience"},
    {"name": "P99 Latency",         "target": "< 8000ms", "window": "1h",
     "alert_at": "> 10000ms",       "severity": 1,
     "why": "P99 >10s = long-running requests are timing out"},
    {"name": "Error Rate",          "target": "< 1%",     "window": "15m",
     "alert_at": "> 2%",            "severity": 1,
     "why": "2% = 1 in 50 calls failing — likely a deployment issue"},
    {"name": "Daily Token Cost",    "target": "< $50/day","window": "1d",
     "alert_at": "> $40/day",       "severity": 2,
     "why": "Alert at 80% of budget = time to investigate before overspend"},
    {"name": "Hallucination Rate",  "target": "< 5%",     "window": "6h",
     "alert_at": "> 8%",            "severity": 2,
     "why": "Spike in hallucinations signals model drift or prompt regression"},
]

print("LLM Service Level Objectives:")
print("="*65)
for slo in slos:
    sev_icon = "🔴" if slo["severity"] == 1 else "🟡"
    print(f"\n{sev_icon} {slo['name']}")
    print(f"   Target   : {slo['target']} over {slo['window']}")
    print(f"   Alert at : {slo['alert_at']}  (SEV-{slo['severity']})")
    print(f"   Reason   : {slo['why']}")


LLM Service Level Objectives:

🔴 API Availability
   Target: 99.5% over 30d
   Alert at: < 99.0% (SEV-1)

🟡 P50 Latency
   Target: < 2000ms over 1h
   Alert at: > 3000ms (SEV-2)

🔴 P99 Latency
   Target: < 8000ms over 1h
   Alert at: > 10000ms (SEV-1)

🔴 Error Rate
   Target: < 1% over 15m
   Alert at: > 2% (SEV-1)

🟡 Daily Token Cost
   Target: < $50/day
   Alert at: > $40/day (SEV-2)

🟡 Hallucination Rate
   Target: < 5% over 6h
   Alert at: > 8% (SEV-2)


## Step 1b: Burn Rate — How Fast Are You Consuming Your Error Budget?

In [ ]:
# SLO Burn Rate — the math behind multi-window alerting
# burn_rate = how fast you consume monthly error budget
# burn_rate=1  → budget exhausted in exactly 30 days  (no alert needed)
# burn_rate=10 → budget exhausted in 3 days           (page the team!)

SLO_TARGET   = 0.995          # 99.5% availability SLO
error_budget = 1 - SLO_TARGET  # 0.5% per 30-day window

def slo_burn_rate(errors: int, total: int) -> dict:
    error_rate  = errors / max(total, 1)
    burn_rate   = error_rate / error_budget        # relative to budget
    hours_left  = (30 * 24 * (1 - error_rate / error_budget)) / burn_rate if burn_rate > 0 else float("inf")
    severity    = ("SEV-1" if burn_rate >= 14.4 else   # budget gone in <2h
                   "SEV-2" if burn_rate >= 6    else   # budget gone in <5h
                   "SEV-3" if burn_rate >= 2    else   # budget gone in <15 days
                   "OK")
    return {
        "error_rate_pct":  round(error_rate * 100, 3),
        "burn_rate":       round(burn_rate, 2),
        "budget_used_pct": round(min(100, error_rate / error_budget * 100), 1),
        "hours_to_empty":  round(max(0, hours_left), 1),
        "severity":        severity,
    }

scenarios = [
    ("Normal traffic",          2,    1000),
    ("Minor degradation",       6,    1000),
    ("Moderate incident (3%)",  30,   1000),
    ("Severe outage (5%)",      50,   1000),
    ("Half requests failing",   500,  1000),
]

print(f"SLO: {SLO_TARGET*100}%  |  Error budget: {error_budget*100}%/30 days")
print("="*75)
print(f"  {'Scenario':<28}  {'Error%':>6}  {'BurnRate':>9}  {'BudgetUsed':>10}  {'ExhaustIn':>10}  Severity")
print("-"*75)
for name, errors, total in scenarios:
    r = slo_burn_rate(errors, total)
    icon = "✅" if r["severity"]=="OK" else ("🟡" if "3" in r["severity"] else "🚨")
    print(f"  {icon} {name:<27} {r['error_rate_pct']:>6.2f}%  "
          f"{r['burn_rate']:>9.1f}x  {r['budget_used_pct']:>9.1f}%  "
          f"{r['hours_to_empty']:>9.1f}h  {r['severity']}")

print()
print("Google SRE multi-window thresholds:")
print("  burn_rate >= 14.4 → SEV-1  fire immediately   (1h + 5min eval windows)")
print("  burn_rate >=  6.0 → SEV-2  page within 5min   (6h + 30min eval windows)")
print("  burn_rate >=  2.0 → SEV-3  ticket within 1h   (24h + 2h eval windows)")
print()
print("Why burn_rate beats raw error rate:")
print("  A 1.5% error rate sounds small — but at burn_rate=3x it exhausts")
print("  your monthly budget in just 10 days. You'd miss the SLO and not know it.")


SLO: 99.5%  |  Error budget: 0.5%/30 days
  Scenario                      Error%  BurnRate  BudgetUsed  ExhaustIn  Severity
---------------------------------------------------------------------------
  ✅ Normal traffic               0.20%       0.4x       40.0%    Infinite  OK
  🟡 Minor degradation            0.60%       1.2x      120.0%     600.0h  OK
  🚨 Moderate incident (3%)       3.00%       6.0x      600.0%      60.0h  SEV-2
  🚨 Severe outage (5%)           5.00%      10.0x     1000.0%      0.0h   SEV-1
  🚨 Half requests failing       50.00%     100.0x     1000.0%      0.0h   SEV-1

Google SRE multi-window thresholds:
  burn_rate >= 14.4 → SEV-1  fire immediately   (1h + 5min eval windows)
  burn_rate >=  6.0 → SEV-2  page within 5min   (6h + 30min eval windows)
  burn_rate >=  2.0 → SEV-3  ticket within 1h   (24h + 2h eval windows)

Why burn_rate beats raw error rate:
  A 1.5% error rate sounds small — but at burn_rate=3x it exhausts
  your monthly budget in just 10 days. You'd 

## Step 2: Create Alert Rules + Budget Alert

In [ ]:
RG             = 'rg-llm-demo'
SUB_ID         = '<your-subscription-id>'
APP_INSIGHTS   = 'appi-llm-demo'
ACTION_GROUP   = 'ag-llm-oncall'
ONCALL_EMAIL   = 'oncall@yourcompany.com'

APP_INSIGHTS_ID = f'/subscriptions/{SUB_ID}/resourceGroups/{RG}/providers/microsoft.insights/components/{APP_INSIGHTS}'
ACTION_GROUP_ID = f'/subscriptions/{SUB_ID}/resourceGroups/{RG}/providers/microsoft.insights/actionGroups/{ACTION_GROUP}'
SUBSCRIPTION_SCOPE = f'/subscriptions/{SUB_ID}'

steps = [
    ('1. Create action group (email + webhook)',
     f'az monitor action-group create --name {ACTION_GROUP} --resource-group {RG} --short-name oncall --action email eng {ONCALL_EMAIL}'),

    ('2. Alert: error rate > 2% (SEV-1)',
     f'az monitor scheduled-query create --name alert-error-rate --resource-group {RG} --scopes {APP_INSIGHTS_ID} --condition-query "customEvents | where name==\'llm_response\' | summarize err=countif(customDimensions.success==\'false\'), tot=count() | extend pct=err*100.0/tot | where pct>2" --severity 1 --evaluation-frequency 5m --window-size 15m --action {ACTION_GROUP_ID}'),

    ('3. Alert: P99 latency > 10s (SEV-1)',
     f'az monitor scheduled-query create --name alert-p99-latency --resource-group {RG} --scopes {APP_INSIGHTS_ID} --condition-query "customEvents | where name==\'llm_response\' | extend lat=todouble(customDimensions.latency_ms) | summarize p99=percentile(lat,99) | where p99>10000" --severity 1 --evaluation-frequency 5m --window-size 30m --action {ACTION_GROUP_ID}'),

    ('4. Budget alert: $40/day warning (80% of $50 SLO)',
     f'az consumption budget create --budget-name budget-llm-daily --resource-group {RG} --amount 50 --time-grain Monthly --time-period-start 2024-01-01T00:00:00Z --time-period-end 2025-01-01T00:00:00Z --notifications warn80/threshold=80/contact-emails={ONCALL_EMAIL}'),

    ('5. Smart detection: anomalous failure rate',
     f'az monitor smart-detector-alert-rule create --name SmartDetect-LLM --resource-group {RG} --scope {APP_INSIGHTS_ID} --detector-id FailureAnomaliesDetector --severity Sev1 --action-groups {ACTION_GROUP_ID}'),

    ('6. Maintenance window (suppress alerts during deployments)',
     f'az monitor scheduled-query update --name alert-error-rate --resource-group {RG} --disabled true   # disable before deploy; re-enable after'),
]

print('='*70)
print('  Alert Rules + Budget Alert Configuration')
print('='*70)
for title, cmd in steps:
    print(f'\n--- {title} ---')
    print(cmd)

print('\n[SYNTHETIC DEMO OUTPUT]')
print('Action group ag-llm-oncall created')
print('Alert alert-error-rate created (SEV-1, fires if >2% errors in 15m)')
print('Alert alert-p99-latency created (SEV-1, fires if P99>10s in 30m)')
print('Budget alert created: email when spend reaches 80% of $50/day')
print('Smart anomaly detector enabled: fires on unusual failure rate spikes')
print('\n[SYNTHETIC DEMO] 14:32 UTC — alert-error-rate FIRED')
print('  Error rate = 4.2% (threshold: 2%)')
print('  Email → oncall@yourcompany.com')
print('  PagerDuty incident #INC-20481 opened')


  Alert Rules + Budget Alert Configuration

--- 1. Create action group ---
az monitor action-group create ...

--- 2. Alert: error rate > 2% ---
az monitor scheduled-query create --name alert-error-rate ...

--- 3. Alert: P99 latency > 10s ---
az monitor scheduled-query create --name alert-p99-latency ...

--- 4. Budget alert: $40/day warning ---
az consumption budget create ...

--- 5. Smart detection ---
az monitor smart-detector-alert-rule create ...

[SYNTHETIC DEMO OUTPUT]
Alert alert-error-rate created (SEV-1)
Alert alert-p99-latency created (SEV-1)
Budget alert created: email at 80% of $50/day

[SYNTHETIC DEMO] 14:32 UTC — alert-error-rate FIRED
  Error rate = 4.2% → PagerDuty #INC-20481 opened


## Step 3: Incident Response Runbook with Rollback Commands

In [ ]:
runbook_steps = {
    "SEV-1 Response Timeline": {
        "0-5 min":  "Acknowledge in PagerDuty. Post in #incidents. Assign IC.",
        "5-15 min": "Diagnose: check App Insights Live Metrics, container logs.",
        "15-30 min":"Mitigate: rollback, scale up, or switch fallback model.",
        "30-60 min":"Communicate: update status page every 15 min.",
        "Post":     "Blameless post-mortem within 48h.",
    },
    "Diagnosis Commands": {
        "Live metrics":    "portal.azure.com → App Insights appi-llm-demo → Live Metrics",
        "Container logs":  "az containerapp logs show --name llm-api-app --resource-group rg-llm-demo --tail 100",
        "Recent errors KQL":"customEvents | where name=='llm_response' and customDimensions.success=='false' | top 20 by timestamp desc",
        "Azure status":    "https://status.azure.com",
    },
    "Mitigation Playbook": {
        "Option A — Rollback":       "az containerapp ingress traffic set --name llm-api-app --resource-group rg-llm-demo --revision-weight llm-api-app--v1=100",
        "Option B — Scale up":       "az containerapp update --name llm-api-app --resource-group rg-llm-demo --min-replicas 3 --max-replicas 20",
        "Option C — Fallback model": "az containerapp update --set-env-vars MODEL_NAME=gpt-35-turbo  (cheaper/faster, less likely to rate-limit)",
        "Option D — Circuit break":  "az containerapp update --set-env-vars FALLBACK_TO_CACHE=true  (serve cached responses while LLM is degraded)",
    }
}

print("LLM API INCIDENT RESPONSE RUNBOOK v1.0")
print("="*65)
for section, items in runbook_steps.items():
    print(f"\n{section}:")
    print("-"*50)
    for k, v in items.items():
        print(f"  {k:<22}: {v}")

print("\n📌 Key Takeaways:")
print("  • Alert at 80% of SLO target — gives time to act BEFORE breach")
print("  • Budget alert at $40 prevents surprise $500 Azure bill")
print("  • Smart anomaly detection catches unusual spikes you didn't think to alert on")
print("  • Maintenance windows = disable alerts during planned deployments")
print("  • Always have 4 mitigation options: rollback, scale, fallback model, cache")


LLM API INCIDENT RESPONSE RUNBOOK v1.0

SEV-1 Response Timeline:
  0-5 min   : Acknowledge in PagerDuty. Post in #incidents. Assign IC.
  5-15 min  : Diagnose: check App Insights Live Metrics.
  15-30 min : Mitigate: rollback, scale up, or switch fallback model.

Mitigation Playbook:
  Option A: az containerapp ingress traffic set ... --revision-weight v1=100
  Option B: az containerapp update --min-replicas 3 --max-replicas 20
  Option C: MODEL_NAME=gpt-35-turbo  (faster fallback)
  Option D: FALLBACK_TO_CACHE=true  (serve cached while LLM degraded)

📌 Key Takeaways:
  • Alert at 80% of SLO target — acts BEFORE breach
  • Budget alert at $40 prevents surprise Azure bills
  • Smart anomaly detection catches spikes you didn't think to alert on
